In the quick start we'll show you know to build a simple LLM application with LangChain.This application will translate text from English into another- it's just a single LLM call plus some prompting. Still this is a great way to get started with LangChain - a lot feature can be built with just some prompting and LLM call!

after seeing this video , you'll have a high level overview of:
<li>Using Language models</li>
<li>Using Prompt Templates and OutputParsers</li>
<li>Using Langchain Expression Language(LCEL) to chain components  together</li>
<li>Debugging and tracing your application using LangSmith </li>
<li>Deploying your application with LangServe</li>

In this section we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.
Note that this chat that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:
<li>Conversational RAG: Enable a chatbot experience over an external source of data</li>
<li>Agents: Build a chatbot that can take actions</li>


In [85]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [86]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "Qwen/Qwen3.6-27B" , groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '0.3.27'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001AC0FBCD660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AC0FBCEF20>, model_name='Qwen/Qwen3.6-27B', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [87]:
from langchain_core.messages import HumanMessage
result = model.invoke([HumanMessage(content="Hi , my name is Muskan Chauhan and AI Forward Deployed Engineer")])
result

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Name: Muskan Chauhan\n   - Role: AI Forward Deployed Engineer\n   - Greeting: "Hi"\n\n2.  **Identify Key Elements:**\n   - Personal introduction (name + professional title)\n   - No specific question or request yet\n   - Professional context (AI, engineering, forward-deployed role implies working directly with customers/partners to implement AI solutions)\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Recognize the professional role\n   - Express enthusiasm/readiness to assist\n   - Ask how I can help (open-ended but tailored to their role)\n   - Keep it professional, concise, and engaging\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! It\'s great to meet you. As an AI Forward Deployed Engineer, you\'re likely working at the intersection of cutting-edge AI and real-world customer solutions—sounds like fascinating work. How can I assist you today?

In [88]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Name: Muskan Chauhan\n   - Role: AI Forward Deployed Engineer\n   - Greeting: "Hi"\n\n2.  **Identify Key Elements:**\n   - Personal introduction (name + professional title)\n   - No specific question or request yet\n   - Professional context (AI, engineering, forward-deployed role implies working directly with customers/partners to implement AI solutions)\n\n3.  **Determine Response Goals:**\n   - Acknowledge and greet by name\n   - Recognize the professional role\n   - Express enthusiasm/readiness to assist\n   - Ask how I can help (open-ended but tailored to their role)\n   - Keep it professional, concise, and engaging\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Muskan! It\'s great to meet you. As an AI Forward Deployed Engineer, you\'re likely working at the intersection of cutting-edge AI and real-world customer solutions—sounds like fascinating work. How can I assist you today? Whether it\'s tro

In [89]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi , My name is Muskan Chauhan "),
    AIMessage(content="Hi Muskan! Nice to meet you. As an AI Forward Deployed Engineer, you\'re clearly working at the exciting intersection of AI development and real-world customer impact. How can I assist you today? Whether it\'s debugging ML pipelines, optimizing model deployments, exploring new AI frameworks, or brainstorming solutions for enterprise AI use cases, I\'m here to help."),
    HumanMessage(content="What kind of salary i am expected in FDE role")
])

 


AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user (Muskan Chauhan) is asking about the expected salary for an "FDE role" (Forward Deployed Engineer). She mentioned her name and I assumed she\'s an AI FDE based on my previous response, but I should clarify that FDE roles exist across tech companies, not just AI-focused ones.\n\n2.  **Identify Key Variables**: Salary for an FDE role depends on several factors:\n   - Location/City/Country\n   - Company (Big Tech vs. Startup vs. Mid-size)\n   - Experience Level (Entry, Mid, Senior, Staff, Principal)\n   - Specific Domain (AI/ML FDE vs. General Software FDE)\n   - Compensation structure (Base salary, bonus, RSUs/stock, benefits)\n\n3.  **Gather General Market Data (as of 2024/2025)**:\n   - FDE is a specialized role, often associated with companies like Meta, Google, Microsoft, Amazon, etc.\n   - In the US (especially tech hubs like SF, Seattle, NYC, Bay Area):\n     - Entry/Junior (0-2 yrs

### Message History
We can use a Message Hsitory class to wrap our model and male it stateful.This will Keep track of inputs and outputs of the model and store them in some datastore. Future Interaction will then load those message and pass them into chain as pat of the input , Let's see how to use this !

In [90]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}


def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model , get_session_history)


e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [91]:
config = {"configurable":{"session_id":"chat1"}}

In [92]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer")
    ],
    config=config
)

In [93]:
## Chnage the config  --> session id
config1 = {"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke(
    [
        HumanMessage(content="Whats my name")
    ],
    config=config1
)
response.content


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "Whats my name"\n2.  **Check Context/History:** In the very first message, the user said: "Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer"\n3.  **Extract Information:** The user\'s name is explicitly stated as "Muskan Chauhan"\n4.  **Formulate Response:** Acknowledge the name clearly and politely. Keep it concise. Example: "Your name is Muskan Chauhan!"\n5.  **Self-Correction/Verification:** Double-check the exact spelling from the prompt: "Muskan Chauhan". Matches. No ambiguity. Ready to respond.✅\n</think>\n\nYour name is **Muskan Chauhan**! How can I help you today? 😊'

In [94]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer")
    ],
    config=config
)

### Prompt Templates 
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case , the raw user input in just a message , which we are passing to the LLM. Let's now make that a bit more complicated. FFisrt , let's add in a system messsagewith some custom instructions (But still taking message as input.) Next , we'll add in more input besides just the message.

In [95]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the nest of your ability."),
        MessagesPlaceholder(variable_name="message")
    ]

)

chain = prompt|model

In [96]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [97]:
config = {
    "configurable": {
        "session_id": "chat1"
    }
}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "What's my name?"
2.  **Check Context:** In the conversation history, the user explicitly stated: "Hi , My Name is Muskan Chauhan and I am a Foward Deployed Engineer" (twice, actually).
3.  **Identify Key Information:** The user's name is Muskan Chauhan.
4.  **Formulate Response:** Acknowledge the question directly and confirm the name clearly. Keep it friendly and concise.
5.  **Draft Response:** Your name is Muskan Chauhan! How can I help you today?
6.  **Self-Correction/Verification:** Matches the context exactly. No extra fluff needed. Ready.✅
</think>

Your name is **Muskan Chauhan**! Let me know how I can help you today. 😊


In [98]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistent.Answer All question to the best of your ability is {language}"
        ),
        MessagesPlaceholder(variable_name='message')
    ]
)

chain = prompt | model

In [99]:
response = chain.invoke({"message": [HumanMessage(content="Hi My name is Muskan Chauhan")],
                        "language":"Hindi"})
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Muskan Chauhan"
   - Language: English
   - Key information: Name is Muskan Chauhan
   - Implicit request: Greeting/Introduction

2.  **Identify Constraints:**
   - System prompt: "Answer All question to the best of your ability is Hindi"
   - This means I must respond in Hindi, regardless of the input language.

3.  **Formulate Response (Internal Translation & Adaptation):**
   - Acknowledge the greeting
   - Use the name provided
   - Respond in Hindi as instructed
   - Keep it polite and helpful
   - Example: "नमस्ते मुस्कन चौहान! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता हूँ?"

4.  **Check Against Constraints:**
   - Is it in Hindi? Yes.
   - Does it acknowledge the name? Yes.
   - Is it helpful and polite? Yes.
   - Does it follow "Answer All question to the best of your ability is Hindi"? Yes.

5.  **Refine Response (Hindi):**
   नमस्ते मुस्कन चौहान! आपका स्वागत है। मैं आपकी कैसे मदद क

Let's now wrap this more complicated chain in a Message Histiry class. This time, becuase there are multiple keys in the input , we need to specify the correct key to use to save the chat history 

In [100]:
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="message"
)

e:\GenerativeAi_python\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [101]:
config = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    {"message":[HumanMessage(content="Hi I am Muskan")],
     "language":"Hindi"},
     config=config
)

print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi I am Muskan"
   - Language: English
   - Intent: Greeting and introducing themselves
   - Constraint from system prompt: "Answer All question to the best of your ability is Hindi"

2.  **Identify Key Requirements:**
   - Respond in Hindi (as per system instruction)
   - Acknowledge the greeting and introduction
   - Maintain a friendly, helpful tone
   - Keep it concise

3.  **Draft Response (Mental Refinement in Hindi):**
   - "नमस्ते मuskan! आपका स्वागत है। बताइए, मैं आपकी कैसे मदद कर सकता हूँ?"
   - Check: Does it match the requirement? Yes, it's in Hindi, acknowledges the name, and offers help.
   - Tone: Polite and helpful.
   - Grammar/Spelling: Correct Hindi.

4.  **Final Output Generation:** (matches the draft)
   "नमस्ते मुस्कान! आपका स्वागत है। बताइए, आज मैं आपकी कैसे सहायता कर सकता हूँ?" (Slightly refined for natural flow)

   All constraints met. Ready.✅
</think>

नमस्ते मुस्कान! आपका स्वाग

## Managing the Conversation History

One import concept to unerstand when building chatsbot is how to manage conversation history. if left unmanaged the list of message will grow unbounded and potentially overflow the context windows of the LLM.. Therefore it is important to add a step that limits the size of the messages you are passing in...................